## Task 1: Probability calibration on a toy dataset

#### Goal: Implement temperature scaling calibration for a neural network classifier on a synthetic dataset.

1. Generate a classification toy dataset using `make_classification` (5000 samples, 20 features). Convert to `float32` tensors and split into train/val, calibration, and test subsets.
2. Train a neural network classifier using cross-validation (5 folds). Save the best model checkpoint per fold — this produces an ensemble of 5 models.
3. Implement a `TemperatureScaler` module that wraps a trained model and divides its logits by a learnable scalar parameter `T`. Implement `fit_temperature` that optimizes `T` using `LBFGS` and `CrossEntropyLoss` on the calibration set. Fit a separate scaler for each of the 5 ensemble models.
4. Compute and plot calibration curves (fraction of positives vs. mean predicted probability, 10 bins) for each model before and after temperature scaling using `calibration_curve` from scikit-learn.
5. Compute Brier score before and after calibration using `brier_score_loss` for each ensemble model.

**Assignment:** Calibrate a classifier on the toy dataset using temperature scaling. Present results as calibration curves and Brier score values.


In [49]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
import torch
from utils import *

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [51]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=5000, n_features=20)
X, y = torch.tensor(X, dtype=torch.float), torch.tensor(y)

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.2)

In [52]:
model = MLP(input_size=20, hidden_layers=[64, 256, 64], output_size=2)

In [53]:
fold_models = []

fold_models = fold_train(X_trainval=X_trainval, y_trainval=y_trainval,
                         model=model, EPOCHS=2, device=device)

Fold 1/5 — best val loss: 0.5819
Fold 2/5 — best val loss: 0.5861
Fold 3/5 — best val loss: 0.5932
Fold 4/5 — best val loss: 0.5874
Fold 5/5 — best val loss: 0.5921
